In [67]:
import pandas as pd
import sys

sys.path.append('..')  # allows Python to look one directory up (the repo root)
from utils import RANDOM_SEED, set_seed

df = pd.read_csv("D:/CitrusBits/pythonic-rebirth/datasets/electricity.csv", low_memory=False)

print(df[['SMPEA', 'SMPEP2']].describe())
print(df[['SMPEA', 'SMPEP2']].head(10))

        SMPEA SMPEP2
count   38014  38014
unique   7340   7814
top     49.56  42.04
freq       63     55
   SMPEA SMPEP2
0  49.26  54.32
1  49.26  54.23
2  49.10  54.23
3  48.04  53.47
4  33.75  39.87
5  33.75  39.87
6  33.75  39.87
7  33.75  39.87
8  33.75  39.87
9  33.75  39.87


In [68]:
print(df['SMPEA'].dtype)
print(df['SMPEP2'].dtype)



str
str


In [69]:
import numpy as np

cols_with_placeholder = ['ForecastWindProduction', 'SystemLoadEA', 'SMPEA',
                         'ORKTemperature', 'ORKWindspeed', 'CO2Intensity',
                         'ActualWindProduction', 'SystemLoadEP2', 'SMPEP2']

for col in cols_with_placeholder:
    if not pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].replace('?', np.nan)
        df[col] = pd.to_numeric(df[col])

print(df[cols_with_placeholder].dtypes)

ForecastWindProduction    float64
SystemLoadEA              float64
SMPEA                     float64
ORKTemperature            float64
ORKWindspeed              float64
CO2Intensity              float64
ActualWindProduction      float64
SystemLoadEP2             float64
SMPEP2                    float64
dtype: object


In [70]:
print(df[['SMPEA', 'SMPEP2']].corr())

           SMPEA    SMPEP2
SMPEA   1.000000  0.617151
SMPEP2  0.617151  1.000000


In [71]:
print("Rows before dropping:", len(df))
df = df.dropna(subset=cols_with_placeholder)
print("Rows after dropping:", len(df))

df['Holiday'] = df['Holiday'].fillna('None')
print(df['Holiday'].value_counts().head())

Rows before dropping: 38014
Rows after dropping: 37682
Holiday
None                36266
Christmas Eve         143
New Year's Eve        143
St Stephen's Day      131
New Year's Day         96
Name: count, dtype: int64


In [72]:
df['DateTime'] = pd.to_datetime(df['DateTime'], format='%d/%m/%Y %H:%M')
print(df['DateTime'].head())
print(df['DateTime'].dtype)

0   2011-11-01 00:00:00
1   2011-11-01 00:30:00
2   2011-11-01 01:00:00
3   2011-11-01 01:30:00
4   2011-11-01 02:00:00
Name: DateTime, dtype: datetime64[us]
datetime64[us]


In [73]:
sample = df.iloc[0]
print("DateTime:", sample['DateTime'])
print("Day:", sample['Day'], "Month:", sample['Month'], "Year:", sample['Year'])
print("DayOfWeek:", sample['DayOfWeek'], "PeriodOfDay:", sample['PeriodOfDay'])

DateTime: 2011-11-01 00:00:00
Day: 1 Month: 11 Year: 2011
DayOfWeek: 1 PeriodOfDay: 0


In [74]:
# encoding holidays now

# we will be doing 1 hot encoding

from sklearn.preprocessing import OneHotEncoder
import pandas as pd

print(df['Holiday'].nunique())  # it is 15

15


In [75]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

ohe_holiday = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

holiday_encoded = ohe_holiday.fit_transform(df[['Holiday']])

holiday_encoded_df = pd.DataFrame(
    holiday_encoded,
    columns=ohe_holiday.get_feature_names_out(['Holiday']),
    index=df.index
)

df = df.drop('Holiday', axis=1)
df = pd.concat([df, holiday_encoded_df], axis=1)

print(df.shape)

(37682, 31)


In [76]:
print(df.groupby('HolidayFlag')['Holiday_None'].value_counts()) if 'Holiday_None' in df.columns else print(
    "check manually")

HolidayFlag  Holiday_None
0            1.0             36266
1            0.0              1416
Name: count, dtype: int64


In [77]:
# since both are duplicates/ doing the same work dropping one
df = df.drop('HolidayFlag', axis=1)
print(df.shape)  # should now be (37682, 30)

(37682, 30)


In [78]:
df = df.drop('DateTime', axis=1)
print(df.shape)

(37682, 29)


In [79]:
print(df.dtypes)

DayOfWeek                         int64
WeekOfYear                        int64
Day                               int64
Month                             int64
Year                              int64
PeriodOfDay                       int64
ForecastWindProduction          float64
SystemLoadEA                    float64
SMPEA                           float64
ORKTemperature                  float64
ORKWindspeed                    float64
CO2Intensity                    float64
ActualWindProduction            float64
SystemLoadEP2                   float64
SMPEP2                          float64
Holiday_Christmas               float64
Holiday_Christmas Eve           float64
Holiday_Easter                  float64
Holiday_Easter Monday           float64
Holiday_Good Friday             float64
Holiday_Holy Saturday           float64
Holiday_June Bank Holiday       float64
Holiday_May Day                 float64
Holiday_New Year's Day          float64
Holiday_New Year's Eve          float64


In [80]:
print(df[['SystemLoadEA', 'SystemLoadEP2']].corr())
print(df[['ForecastWindProduction', 'ActualWindProduction']].corr())

               SystemLoadEA  SystemLoadEP2
SystemLoadEA       1.000000       0.972738
SystemLoadEP2      0.972738       1.000000
                        ForecastWindProduction  ActualWindProduction
ForecastWindProduction                1.000000              0.953812
ActualWindProduction                  0.953812              1.000000


In [81]:
X = df.drop(['SMPEP2', 'SystemLoadEP2', 'ActualWindProduction'], axis=1)
y = df['SMPEP2']

print(X.shape)
print(X.columns.tolist())

(37682, 26)
['DayOfWeek', 'WeekOfYear', 'Day', 'Month', 'Year', 'PeriodOfDay', 'ForecastWindProduction', 'SystemLoadEA', 'SMPEA', 'ORKTemperature', 'ORKWindspeed', 'CO2Intensity', 'Holiday_Christmas', 'Holiday_Christmas Eve', 'Holiday_Easter', 'Holiday_Easter Monday', 'Holiday_Good Friday', 'Holiday_Holy Saturday', 'Holiday_June Bank Holiday', 'Holiday_May Day', "Holiday_New Year's Day", "Holiday_New Year's Eve", 'Holiday_None', 'Holiday_October Bank Holiday', "Holiday_St Patrick's Day", "Holiday_St Stephen's Day"]


In [82]:
from sklearn.model_selection import train_test_split

X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.20, random_state=RANDOM_SEED)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.125, random_state=RANDOM_SEED)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

Train: (26376, 26) Val: (3769, 26) Test: (7537, 26)


In [83]:
from sklearn.preprocessing import StandardScaler

numeric_cols = ['DayOfWeek', 'WeekOfYear', 'Day', 'Month', 'Year', 'PeriodOfDay',
                'ForecastWindProduction', 'SystemLoadEA', 'SMPEA', 'ORKTemperature',
                'ORKWindspeed', 'CO2Intensity']

scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_val[numeric_cols] = scaler.transform(X_val[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

print(X_train[numeric_cols].describe())

          DayOfWeek    WeekOfYear           Day         Month          Year  \
count  2.637600e+04  2.637600e+04  2.637600e+04  2.637600e+04  2.637600e+04   
mean  -3.852275e-17 -4.148604e-17  7.165771e-17 -2.262875e-17  1.732376e-13   
std    1.000019e+00  1.000019e+00  1.000019e+00  1.000019e+00  1.000019e+00   
min   -1.502692e+00 -1.735361e+00 -1.673152e+00 -1.646203e+00 -2.218020e+00   
25%   -1.001763e+00 -8.990946e-01 -8.784315e-01 -8.049533e-01 -6.150801e-01   
50%    9.495926e-05  1.499918e-03  2.982053e-02  3.629594e-02 -6.150801e-01   
75%    1.001953e+00  9.664226e-01  8.245411e-01  8.775452e-01  9.878596e-01   
max    1.502882e+00  1.545376e+00  1.732793e+00  1.438378e+00  9.878596e-01   

        PeriodOfDay  ForecastWindProduction  SystemLoadEA         SMPEA  \
count  2.637600e+04            2.637600e+04  2.637600e+04  2.637600e+04   
mean  -6.411479e-17            5.387798e-17  8.240636e-16  1.258051e-16   
std    1.000019e+00            1.000019e+00  1.000019e+00  1.00

In [84]:
from sklearn.preprocessing import StandardScaler
import torch

y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).flatten()
y_val_scaled = y_scaler.transform(y_val.values.reshape(-1, 1)).flatten()

y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32).view(-1, 1)
y_val_tensor = torch.tensor(y_val_scaled, dtype=torch.float32).view(-1, 1)

print(y_train_scaled.mean(), y_train_scaled.std())  # should be ~0 and ~1 now


1.7564219886092861e-16 0.9999999999999999


In [85]:
import torch.nn as nn
import os

if os.path.exists(torch_model_path):
    os.remove(torch_model_path)

torch.manual_seed(RANDOM_SEED)
lr_model_torch = LinearRegressionModel(X_train.shape[1])

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(lr_model_torch.parameters(), lr=0.01)

X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val.values, dtype=torch.float32)

epochs = 1000
train_losses = []
val_losses = []

for epoch in range(epochs):
    lr_model_torch.train()
    optimizer.zero_grad()
    predictions = lr_model_torch(X_train_tensor)
    loss = criterion(predictions, y_train_tensor)
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())

    lr_model_torch.eval()
    with torch.no_grad():
        val_preds = lr_model_torch(X_val_tensor)
        val_loss = criterion(val_preds, y_val_tensor)
        val_losses.append(val_loss.item())

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch + 1}/{epochs} | Train Loss: {loss.item():.4f} | Val Loss: {val_loss.item():.4f}")

torch.save(lr_model_torch.state_dict(), torch_model_path)
print("Model trained and saved.")

Epoch 100/1000 | Train Loss: 0.5506 | Val Loss: 0.5849
Epoch 200/1000 | Train Loss: 0.5506 | Val Loss: 0.5849
Epoch 300/1000 | Train Loss: 0.5506 | Val Loss: 0.5850
Epoch 400/1000 | Train Loss: 0.5505 | Val Loss: 0.5850
Epoch 500/1000 | Train Loss: 0.5505 | Val Loss: 0.5850
Epoch 600/1000 | Train Loss: 0.5505 | Val Loss: 0.5851
Epoch 700/1000 | Train Loss: 0.5505 | Val Loss: 0.5851
Epoch 800/1000 | Train Loss: 0.5505 | Val Loss: 0.5851
Epoch 900/1000 | Train Loss: 0.5505 | Val Loss: 0.5851
Epoch 1000/1000 | Train Loss: 0.5505 | Val Loss: 0.5851
Model trained and saved.


In [86]:
from sklearn.metrics import mean_squared_error, r2_score

lr_model_torch.eval()
with torch.no_grad():
    val_preds_scaled = lr_model_torch(X_val_tensor).numpy().flatten()

val_preds_actual = y_scaler.inverse_transform(val_preds_scaled.reshape(-1, 1)).flatten()

lr_val_mse_torch = mean_squared_error(y_val, val_preds_actual)
lr_val_r2_torch = r2_score(y_val, val_preds_actual)

print(f"PyTorch Linear Regression (Validation) → MSE: {lr_val_mse_torch:.4f}, R²: {lr_val_r2_torch:.4f}")

PyTorch Linear Regression (Validation) → MSE: 731.4629, R²: 0.3929


In [87]:
from sklearn.tree import DecisionTreeRegressor

dt_model = DecisionTreeRegressor(random_state=RANDOM_SEED, max_depth=8)
dt_model.fit(X_train, y_train)

dt_val_preds = dt_model.predict(X_val)
dt_val_mse = mean_squared_error(y_val, dt_val_preds)
dt_val_r2 = r2_score(y_val, dt_val_preds)

print(f"{'Model':<25}{'MSE':<15}{'R²':<10}")
print(f"{'PyTorch Linear Regression':<25}{731.4629:<15.4f}{0.3929:<10.4f}")
print(f"{'Decision Tree':<25}{dt_val_mse:<15.4f}{dt_val_r2:<10.4f}")

Model                    MSE            R²        
PyTorch Linear Regression731.4629       0.3929    
Decision Tree            632.3058       0.4752    


In [89]:
from sklearn.metrics import r2_score, mean_squared_error

# Decision Tree (sklearn)
dt_test_preds = dt_model.predict(X_test)
dt_test_mse = mean_squared_error(y_test, dt_test_preds)
dt_test_r2 = r2_score(y_test, dt_test_preds)

# Linear Regression (PyTorch)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
lr_model_torch.eval()
with torch.no_grad():
    torch_test_preds_scaled = lr_model_torch(X_test_tensor).numpy().flatten()
torch_test_preds = y_scaler.inverse_transform(torch_test_preds_scaled.reshape(-1, 1)).flatten()
torch_test_mse = mean_squared_error(y_test, torch_test_preds)
torch_test_r2 = r2_score(y_test, torch_test_preds)

print(f"{'Model':<30}{'Test MSE':<15}{'Test R²':<10}")
print(f"{'PyTorch Linear Regression':<30}{torch_test_mse:<15.4f}{torch_test_r2:<10.4f}")
print(f"{'Decision Tree (sklearn)':<30}{dt_test_mse:<15.4f}{dt_test_r2:<10.4f}")

Model                         Test MSE       Test R²   
PyTorch Linear Regression     788.8098       0.3927    
Decision Tree (sklearn)       813.3658       0.3738    


In [90]:
importances = pd.Series(dt_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(importances.head(10))

SMPEA                     0.602845
SystemLoadEA              0.197752
PeriodOfDay               0.054993
ForecastWindProduction    0.035057
CO2Intensity              0.033609
ORKTemperature            0.025107
DayOfWeek                 0.012715
WeekOfYear                0.011608
ORKWindspeed              0.007602
Day                       0.005576
dtype: float64


**Summary**

PyTorch Linear Regression achieved a Test R² of 0.3927, while Decision Tree
achieved 0.3738. Interestingly, Decision Tree outperformed Linear Regression
on the validation set (0.4752 vs 0.3929) but underperformed on the held-out
test set — suggesting some overfitting to the validation split at max_depth=8.
This highlights the importance of a separate, untouched test set for honest
model evaluation.

Feature importance analysis showed [X] as the most influential predictor,
consistent with electricity price following [cyclical/threshold] patterns
that Linear Regression's linear coefficients cannot fully capture.